# 1. DCF vs Price From Financial Modeling Prep

This notebook isolates the DCF workflow from the price-target notebook. It includes the FMP DCF snapshot, a transparent annual backfilled DCF build from historical statements, a quarterly TTM-based backfilled DCF build, and a comparison chart against price.

Change `TICKER` in the first code cell and rerun the notebook. Inputs like `BRK\B`, `BRK/B`, and `BRK.B` are normalized automatically for FMP.

In [ ]:
# 2. Setup and FMP helpers
from pathlib import Path
import os

import pandas as pd
import plotly.graph_objects as go
import requests


TICKER = "AAPL"  # Change this to any supported ticker and rerun.
BASE_URL = "https://financialmodelingprep.com/stable"


def normalize_symbol_input(raw_symbol: str) -> str:
    symbol = raw_symbol.strip().upper()
    for separator in ("\\", "/", "."):
        symbol = symbol.replace(separator, "-")
    return symbol


SYMBOL = normalize_symbol_input(TICKER)


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("Could not find the repo .env file.")


def load_key_from_dotenv() -> str:
    dotenv_path = find_repo_root(Path.cwd()) / ".env"
    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        if name.strip() == "FMP_API_KEY":
            return value.strip().strip("\"'")
    raise RuntimeError("FMP_API_KEY was not found in .env")


def get_api_key() -> str:
    return os.getenv("FMP_API_KEY") or load_key_from_dotenv()


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    payload = response.json()

    if response.status_code != 200:
        raise RuntimeError(f"FMP request failed with status {response.status_code}: {payload}")

    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])

    if isinstance(payload, dict) and "value" in payload:
        return payload["value"]

    return payload

In [ ]:
# 3. Load company context and price history
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

dcf_full_price_history = pd.DataFrame(request_json("historical-price-eod/light", symbol=SYMBOL)).copy()
dcf_full_price_history["date"] = pd.to_datetime(dcf_full_price_history["date"], errors="coerce")
dcf_full_price_history["price"] = pd.to_numeric(dcf_full_price_history["price"], errors="coerce")
dcf_full_price_history = dcf_full_price_history.dropna(subset=["date", "price"]).sort_values("date")

In [ ]:
# 4. Plot FMP DCF snapshot vs price
legacy_dcf_response = requests.get(
    f"https://financialmodelingprep.com/api/v3/discounted-cash-flow/{SYMBOL}",
    params={"apikey": get_api_key()},
    timeout=30,
).json()

stable_dcf_response = request_json("discounted-cash-flow", symbol=SYMBOL)

if isinstance(legacy_dcf_response, list):
    legacy_dcf_history = pd.DataFrame(legacy_dcf_response)
elif isinstance(legacy_dcf_response, dict):
    legacy_dcf_history = pd.DataFrame([legacy_dcf_response])
else:
    legacy_dcf_history = pd.DataFrame()

if isinstance(stable_dcf_response, list):
    stable_dcf_history = pd.DataFrame(stable_dcf_response)
elif isinstance(stable_dcf_response, dict):
    stable_dcf_history = pd.DataFrame([stable_dcf_response])
else:
    stable_dcf_history = pd.DataFrame()

dcf_history = legacy_dcf_history if "dcf" in legacy_dcf_history.columns else stable_dcf_history

if dcf_history.empty or "dcf" not in dcf_history.columns:
    raise RuntimeError(
        f"FMP did not return usable DCF data for {SYMBOL}. Legacy payload: {legacy_dcf_response!r}. Stable payload: {stable_dcf_response!r}."
    )

dcf_history["date"] = pd.to_datetime(dcf_history["date"], errors="coerce")
dcf_history["dcf"] = pd.to_numeric(dcf_history["dcf"], errors="coerce")
dcf_history["Stock Price"] = pd.to_numeric(dcf_history.get("Stock Price"), errors="coerce")
dcf_history = dcf_history.dropna(subset=["date", "dcf"]).sort_values("date").reset_index(drop=True)

if dcf_history.empty:
    raise RuntimeError(f"FMP returned DCF rows for {SYMBOL}, but none had usable dates and DCF values.")

dcf_plot = go.Figure()

dcf_plot.add_trace(
    go.Scatter(
        x=dcf_full_price_history["date"],
        y=dcf_full_price_history["price"],
        name=f"{SYMBOL} close",
        line={"color": "#e5eefc", "width": 2.5},
    )
)

dcf_mode = "lines+markers" if len(dcf_history) > 1 else "markers+text"
dcf_text = None if len(dcf_history) > 1 else [f"DCF: {dcf_history['dcf'].iloc[0]:.2f}"]

dcf_plot.add_trace(
    go.Scatter(
        x=dcf_history["date"],
        y=dcf_history["dcf"],
        name="FMP DCF",
        mode=dcf_mode,
        line={"color": "#f472b6", "width": 2.5},
        marker={"color": "#f472b6", "size": 10},
        text=dcf_text,
        textposition="top center",
    )
)

if dcf_history["Stock Price"].notna().any():
    dcf_plot.add_trace(
        go.Scatter(
            x=dcf_history.loc[dcf_history["Stock Price"].notna(), "date"],
            y=dcf_history.loc[dcf_history["Stock Price"].notna(), "Stock Price"],
            name="FMP stock price at DCF snapshot",
            mode="markers",
            marker={"color": "#38bdf8", "size": 8, "symbol": "diamond"},
        )
    )

dcf_plot.update_layout(
    title=f"{chart_label} DCF vs price",
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    xaxis_title="Date",
    yaxis_title="USD per share",
    autosize=True,
    height=680,
    margin={"l": 60, "r": 30, "t": 90, "b": 60},
)

dcf_plot.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)
dcf_plot.update_yaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)

if len(dcf_history) == 1:
    print(
        "FMP returned a single dated DCF snapshot rather than a documented historical DCF series. "
        "The chart shows that snapshot against the full price history."
    )

dcf_plot.show(config={"responsive": True, "displaylogo": False})

FMP returned a single dated DCF snapshot rather than a documented historical DCF series. The chart shows that snapshot against the full price history.


## 5. Historical DCF Inputs From FMP Financial Statements

This block pulls the core annual statement history needed to start building a DCF model from Financial Modeling Prep. It loads income statement, balance sheet, and cash flow history, then assembles a merged table with the operating, tax, reinvestment, working-capital, cash, debt, and share-count fields typically needed for a DCF build.

In [ ]:
# 6. Build annual historical DCF inputs
statement_limit = 20

income_statement_history = pd.DataFrame(
    request_json("income-statement", symbol=SYMBOL, limit=statement_limit)
).copy()
balance_sheet_history = pd.DataFrame(
    request_json("balance-sheet-statement", symbol=SYMBOL, limit=statement_limit)
).copy()
cash_flow_history = pd.DataFrame(
    request_json("cash-flow-statement", symbol=SYMBOL, limit=statement_limit)
).copy()

statement_frames = [income_statement_history, balance_sheet_history, cash_flow_history]
for frame in statement_frames:
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    if "calendarYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["calendarYear"], errors="coerce")
    if "fillingDate" in frame.columns:
        frame["fillingDate"] = pd.to_datetime(frame["fillingDate"], errors="coerce")
    if "filingDate" in frame.columns:
        frame["filingDate"] = pd.to_datetime(frame["filingDate"], errors="coerce")

income_columns = [
    "date",
    "calendarYear",
    "period",
    "revenue",
    "ebitda",
    "operatingIncome",
    "incomeBeforeTax",
    "incomeTaxExpense",
    "netIncome",
    "weightedAverageShsOut",
    "weightedAverageShsOutDil",
]

balance_columns = [
    "date",
    "calendarYear",
    "period",
    "cashAndCashEquivalents",
    "cashAndShortTermInvestments",
    "netReceivables",
    "inventory",
    "otherCurrentAssets",
    "totalCurrentAssets",
    "accountPayables",
    "otherCurrentLiabilities",
    "totalCurrentLiabilities",
    "shortTermDebt",
    "longTermDebt",
    "totalDebt",
    "netDebt",
]

cash_flow_columns = [
    "date",
    "calendarYear",
    "period",
    "depreciationAndAmortization",
    "stockBasedCompensation",
    "changeInWorkingCapital",
    "operatingCashFlow",
    "capitalExpenditure",
    "freeCashFlow",
]

income_dcf_inputs = income_statement_history.reindex(columns=income_columns).rename(
    columns={
        "period": "incomePeriod",
        "depreciationAndAmortization": "incomeDepreciationAndAmortization",
    }
)

balance_dcf_inputs = balance_sheet_history.reindex(columns=balance_columns).rename(
    columns={"period": "balancePeriod"}
)

cash_flow_dcf_inputs = cash_flow_history.reindex(columns=cash_flow_columns).rename(
    columns={
        "period": "cashFlowPeriod",
        "depreciationAndAmortization": "cashFlowDepreciationAndAmortization",
    }
)

historical_dcf_inputs = (
    income_dcf_inputs
    .merge(balance_dcf_inputs, on=["date", "calendarYear"], how="outer")
    .merge(cash_flow_dcf_inputs, on=["date", "calendarYear"], how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

numeric_columns = [
    column
    for column in historical_dcf_inputs.columns
    if column not in {"date", "incomePeriod", "balancePeriod", "cashFlowPeriod"}
    and historical_dcf_inputs[column].dtype == object
]
for column in numeric_columns:
    historical_dcf_inputs[column] = pd.to_numeric(historical_dcf_inputs[column], errors="coerce")

historical_dcf_inputs["effectiveTaxRate"] = (
    historical_dcf_inputs["incomeTaxExpense"]
    .div(historical_dcf_inputs["incomeBeforeTax"].replace(0, pd.NA))
    .clip(lower=0, upper=1)
    .fillna(0)
 )

historical_dcf_inputs["operatingWorkingCapital"] = (
    historical_dcf_inputs["netReceivables"].fillna(0)
    + historical_dcf_inputs["inventory"].fillna(0)
    + historical_dcf_inputs["otherCurrentAssets"].fillna(0)
    - historical_dcf_inputs["accountPayables"].fillna(0)
    - historical_dcf_inputs["otherCurrentLiabilities"].fillna(0)
)
historical_dcf_inputs["changeInOperatingWorkingCapital"] = historical_dcf_inputs["operatingWorkingCapital"].diff()

historical_dcf_inputs["cashFlowDepreciationAndAmortization"] = historical_dcf_inputs[
    "cashFlowDepreciationAndAmortization"
].fillna(0)
historical_dcf_inputs["capitalExpenditure"] = historical_dcf_inputs["capitalExpenditure"].fillna(0)
historical_dcf_inputs["operatingCashFlow"] = historical_dcf_inputs["operatingCashFlow"].fillna(0)
historical_dcf_inputs["freeCashFlow"] = historical_dcf_inputs["freeCashFlow"].fillna(
    historical_dcf_inputs["operatingCashFlow"] + historical_dcf_inputs["capitalExpenditure"]
)

historical_dcf_inputs["ebitAfterTax"] = historical_dcf_inputs["operatingIncome"] * (
    1 - historical_dcf_inputs["effectiveTaxRate"]
)
historical_dcf_inputs["unleveredFreeCashFlowProxy"] = (
    historical_dcf_inputs["ebitAfterTax"]
    + historical_dcf_inputs["cashFlowDepreciationAndAmortization"]
    - historical_dcf_inputs["capitalExpenditure"].abs()
    - historical_dcf_inputs["changeInOperatingWorkingCapital"].fillna(0)
)

historical_dcf_inputs = historical_dcf_inputs[[
    "date",
    "calendarYear",
    "incomePeriod",
    "balancePeriod",
    "cashFlowPeriod",
    "revenue",
    "ebitda",
    "operatingIncome",
    "incomeBeforeTax",
    "incomeTaxExpense",
    "effectiveTaxRate",
    "netIncome",
    "cashFlowDepreciationAndAmortization",
    "stockBasedCompensation",
    "operatingCashFlow",
    "capitalExpenditure",
    "freeCashFlow",
    "netReceivables",
    "inventory",
    "otherCurrentAssets",
    "accountPayables",
    "otherCurrentLiabilities",
    "operatingWorkingCapital",
    "changeInOperatingWorkingCapital",
    "cashAndCashEquivalents",
    "cashAndShortTermInvestments",
    "shortTermDebt",
    "longTermDebt",
    "totalDebt",
    "netDebt",
    "weightedAverageShsOut",
    "weightedAverageShsOutDil",
    "ebitAfterTax",
    "unleveredFreeCashFlowProxy",
]]

## 7. Historical DCF Assumptions and Backfill

This block uses FMP data to seed DCF assumptions, then backfills an approximate historical DCF series from the statement history. It is not FMP's proprietary historical DCF curve; it is a transparent reconstruction using FMP fundamentals, FMP risk inputs, and a simple 5-year projection model for each historical period.

In [ ]:
# 8. Load DCF assumptions
def dcf_as_frame(payload):
    if isinstance(payload, list):
        return pd.DataFrame(payload)
    if isinstance(payload, dict):
        return pd.DataFrame([payload])
    return pd.DataFrame()


def pct_to_decimal(value, default=None):
    if pd.isna(value):
        return default
    numeric_value = float(value)
    return numeric_value / 100 if abs(numeric_value) > 1 else numeric_value


def clamp(value, lower, upper):
    return max(lower, min(upper, value))


def median_or_default(series, default, window=3):
    numeric_series = pd.to_numeric(series, errors="coerce").dropna().tail(window)
    return float(numeric_series.median()) if not numeric_series.empty else default


custom_dcf_inputs = dcf_as_frame(request_json("custom-discounted-cash-flow", symbol=SYMBOL))
profile_snapshot = dcf_as_frame(request_json("profile", symbol=SYMBOL))
treasury_rates = dcf_as_frame(request_json("treasury-rates"))
market_risk_premium_history = dcf_as_frame(request_json("market-risk-premium"))

custom_dcf_row = custom_dcf_inputs.iloc[-1] if not custom_dcf_inputs.empty else pd.Series(dtype=object)
profile_row = profile_snapshot.iloc[0] if not profile_snapshot.empty else pd.Series(dtype=object)
treasury_row = treasury_rates.iloc[0] if not treasury_rates.empty else pd.Series(dtype=object)
market_risk_premium_row = (
    market_risk_premium_history.iloc[0] if not market_risk_premium_history.empty else pd.Series(dtype=object)
)

reference_risk_free_rate = pct_to_decimal(
    custom_dcf_row.get("riskFreeRate", treasury_row.get("year10", treasury_row.get("10Y", 4.0))),
    default=0.04,
)
reference_market_risk_premium = pct_to_decimal(
    custom_dcf_row.get(
        "marketRiskPremium",
        market_risk_premium_row.get("marketRiskPremium", market_risk_premium_row.get("value", 4.72)),
    ),
    default=0.0472,
)
reference_beta = float(custom_dcf_row.get("beta", profile_row.get("beta", 1.0)) or 1.0)
reference_cost_of_equity = pct_to_decimal(
    custom_dcf_row.get("costOfEquity", reference_risk_free_rate + reference_beta * reference_market_risk_premium),
    default=reference_risk_free_rate + reference_beta * reference_market_risk_premium,
)
reference_cost_of_debt = pct_to_decimal(custom_dcf_row.get("costofDebt", 3.64), default=0.0364)
reference_terminal_growth_rate = pct_to_decimal(custom_dcf_row.get("longTermGrowthRate", 4.0), default=0.04)
reference_wacc = pct_to_decimal(custom_dcf_row.get("wacc", pd.NA), default=pd.NA)

dcf_assumptions = pd.DataFrame(
    [
        {
            "symbol": SYMBOL,
            "riskFreeRate": reference_risk_free_rate,
            "marketRiskPremium": reference_market_risk_premium,
            "beta": reference_beta,
            "costOfEquity": reference_cost_of_equity,
            "costOfDebt": reference_cost_of_debt,
            "longTermGrowthRate": reference_terminal_growth_rate,
            "referenceWaccFromFmp": reference_wacc,
        }
    ]
)

In [ ]:
# 9. Build annual backfilled DCF and plot
forecast_years = 5

historical_dcf_base = pd.merge_asof(
    historical_dcf_inputs.sort_values("date"),
    dcf_full_price_history[["date", "price"]].sort_values("date"),
    on="date",
    direction="backward",
).rename(columns={"price": "marketPriceAtDate"})

historical_dcf_rows = []

for index, row in historical_dcf_base.iterrows():
    if pd.isna(row["revenue"]) or row["revenue"] <= 0:
        continue

    lookback = historical_dcf_base.iloc[: index + 1].tail(4).copy()
    revenue_growth_series = lookback["revenue"].pct_change().replace([float("inf"), float("-inf")], pd.NA)
    operating_margin_series = lookback["operatingIncome"].div(lookback["revenue"].replace(0, pd.NA))
    depreciation_margin_series = lookback["cashFlowDepreciationAndAmortization"].div(
        lookback["revenue"].replace(0, pd.NA)
    )
    capex_margin_series = lookback["capitalExpenditure"].abs().div(lookback["revenue"].replace(0, pd.NA))
    working_capital_margin_series = lookback["operatingWorkingCapital"].div(
        lookback["revenue"].replace(0, pd.NA)
    )

    revenue_growth = clamp(median_or_default(revenue_growth_series, 0.05), -0.03, 0.15)
    operating_margin = clamp(
        median_or_default(operating_margin_series, row["operatingIncome"] / row["revenue"]), -0.05, 0.5
    )
    depreciation_margin = clamp(
        median_or_default(depreciation_margin_series, 0.04), 0.0, 0.2
    )
    capex_margin = clamp(median_or_default(capex_margin_series, 0.04), 0.0, 0.2)
    working_capital_margin = median_or_default(
        working_capital_margin_series, row["operatingWorkingCapital"] / row["revenue"]
    )
    effective_tax_rate = clamp(
        median_or_default(lookback["effectiveTaxRate"], reference_risk_free_rate, window=4), 0.0, 0.35
    )

    diluted_shares = row["weightedAverageShsOutDil"]
    if pd.isna(diluted_shares) or diluted_shares <= 0:
        diluted_shares = row["weightedAverageShsOut"]
    if pd.isna(diluted_shares) or diluted_shares <= 0:
        continue

    market_price = row["marketPriceAtDate"]
    market_equity_value = market_price * diluted_shares if pd.notna(market_price) else pd.NA
    total_debt = row["totalDebt"] if pd.notna(row["totalDebt"]) else 0.0
    net_debt = row["netDebt"] if pd.notna(row["netDebt"]) else total_debt
    debt_weight = (
        float(total_debt) / float(total_debt + market_equity_value)
        if pd.notna(market_equity_value) and total_debt + market_equity_value > 0
        else 0.0
    )
    equity_weight = 1 - debt_weight

    after_tax_cost_of_debt = reference_cost_of_debt * (1 - effective_tax_rate)
    wacc = debt_weight * after_tax_cost_of_debt + equity_weight * reference_cost_of_equity
    if not wacc or pd.isna(wacc):
        wacc = reference_cost_of_equity
    terminal_growth_rate = min(reference_terminal_growth_rate, wacc - 0.01) if wacc > 0.02 else 0.01
    terminal_growth_rate = max(terminal_growth_rate, 0.005)

    projected_revenue = float(row["revenue"])
    projected_working_capital = float(row["operatingWorkingCapital"] if pd.notna(row["operatingWorkingCapital"]) else 0.0)
    discounted_cash_flows = []
    projected_ufcf_final_year = pd.NA

    for forecast_year in range(1, forecast_years + 1):
        projected_revenue *= 1 + revenue_growth
        projected_ebit = projected_revenue * operating_margin
        projected_ebit_after_tax = projected_ebit * (1 - effective_tax_rate)
        projected_depreciation = projected_revenue * depreciation_margin
        projected_capex = projected_revenue * capex_margin
        next_working_capital = projected_revenue * working_capital_margin
        projected_change_in_working_capital = next_working_capital - projected_working_capital
        projected_working_capital = next_working_capital
        projected_ufcf = (
            projected_ebit_after_tax
            + projected_depreciation
            - projected_capex
            - projected_change_in_working_capital
        )
        discounted_cash_flows.append(projected_ufcf / ((1 + wacc) ** forecast_year))
        projected_ufcf_final_year = projected_ufcf

    terminal_cash_flow = projected_ufcf_final_year * (1 + terminal_growth_rate)
    terminal_value = terminal_cash_flow / (wacc - terminal_growth_rate)
    present_terminal_value = terminal_value / ((1 + wacc) ** forecast_years)
    enterprise_value = sum(discounted_cash_flows) + present_terminal_value
    equity_value = enterprise_value - net_debt
    dcf_per_share = equity_value / diluted_shares
    premium_discount_pct = dcf_per_share / market_price - 1 if pd.notna(market_price) and market_price else pd.NA

    historical_dcf_rows.append(
        {
            "date": row["date"],
            "calendarYear": row["calendarYear"],
            "revenueGrowthAssumption": revenue_growth,
            "operatingMarginAssumption": operating_margin,
            "taxRateAssumption": effective_tax_rate,
            "wacc": wacc,
            "terminalGrowthRate": terminal_growth_rate,
            "marketPriceAtDate": market_price,
            "marketEquityValue": market_equity_value,
            "totalDebt": total_debt,
            "netDebt": net_debt,
            "enterpriseValue": enterprise_value,
            "equityValue": equity_value,
            "historicalDcfPerShare": dcf_per_share,
            "premiumDiscountPct": premium_discount_pct,
            "reportedFreeCashFlow": row["freeCashFlow"],
            "ufcfProxyAtDate": row["unleveredFreeCashFlowProxy"],
        }
    )

historical_dcf_series = pd.DataFrame(historical_dcf_rows)

historical_dcf_plot = go.Figure()
historical_dcf_plot.add_trace(
    go.Scatter(
        x=dcf_full_price_history["date"],
        y=dcf_full_price_history["price"],
        name=f"{SYMBOL} close",
        line={"color": "#e5eefc", "width": 2.5},
    )
)
historical_dcf_plot.add_trace(
    go.Scatter(
        x=historical_dcf_series["date"],
        y=historical_dcf_series["historicalDcfPerShare"],
        name="Backfilled DCF per share",
        mode="lines+markers",
        line={"color": "#f472b6", "width": 2.5},
        marker={"color": "#f472b6", "size": 9},
    )
)

historical_dcf_plot.update_layout(
    title=f"{chart_label} backfilled DCF vs price",
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    xaxis_title="Date",
    yaxis_title="USD per share",
    autosize=True,
    height=680,
    margin={"l": 60, "r": 30, "t": 90, "b": 60},
)
historical_dcf_plot.update_xaxes(
    showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True
)
historical_dcf_plot.update_yaxes(
    showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True
)

historical_dcf_plot.show(config={"responsive": True, "displaylogo": False})

## 10. Quarterly DCF Build

> This version uses quarterly statements, converts flow items to trailing-twelve-month totals, and values the company from each quarterly reporting date. It keeps balance-sheet items at the quarter-end level while annualizing revenue, operating income, depreciation, capex, and cash flow inputs via TTM aggregation.

In [ ]:
# 11. Build quarterly TTM DCF inputs
quarterly_statement_limit = 40

quarterly_income_columns = income_columns if "income_columns" in globals() else [
    "date",
    "calendarYear",
    "period",
    "revenue",
    "ebitda",
    "operatingIncome",
    "incomeBeforeTax",
    "incomeTaxExpense",
    "netIncome",
    "weightedAverageShsOut",
    "weightedAverageShsOutDil",
]
quarterly_balance_columns = balance_columns if "balance_columns" in globals() else [
    "date",
    "calendarYear",
    "period",
    "cashAndCashEquivalents",
    "cashAndShortTermInvestments",
    "netReceivables",
    "inventory",
    "otherCurrentAssets",
    "totalCurrentAssets",
    "accountPayables",
    "otherCurrentLiabilities",
    "totalCurrentLiabilities",
    "shortTermDebt",
    "longTermDebt",
    "totalDebt",
    "netDebt",
]
quarterly_cash_flow_columns = cash_flow_columns if "cash_flow_columns" in globals() else [
    "date",
    "calendarYear",
    "period",
    "depreciationAndAmortization",
    "stockBasedCompensation",
    "changeInWorkingCapital",
    "operatingCashFlow",
    "capitalExpenditure",
    "freeCashFlow",
]

quarterly_income_statement_history = pd.DataFrame(
    request_json("income-statement", symbol=SYMBOL, period="quarter", limit=quarterly_statement_limit)
).copy()
quarterly_balance_sheet_history = pd.DataFrame(
    request_json("balance-sheet-statement", symbol=SYMBOL, period="quarter", limit=quarterly_statement_limit)
).copy()
quarterly_cash_flow_history = pd.DataFrame(
    request_json("cash-flow-statement", symbol=SYMBOL, period="quarter", limit=quarterly_statement_limit)
).copy()

quarterly_statement_frames = [
    quarterly_income_statement_history,
    quarterly_balance_sheet_history,
    quarterly_cash_flow_history,
]
for frame in quarterly_statement_frames:
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    if "calendarYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["calendarYear"], errors="coerce")
    if "fillingDate" in frame.columns:
        frame["fillingDate"] = pd.to_datetime(frame["fillingDate"], errors="coerce")
    if "filingDate" in frame.columns:
        frame["filingDate"] = pd.to_datetime(frame["filingDate"], errors="coerce")

quarterly_income_dcf_inputs = quarterly_income_statement_history.reindex(columns=quarterly_income_columns).rename(
    columns={"period": "incomePeriod"}
)
quarterly_balance_dcf_inputs = quarterly_balance_sheet_history.reindex(columns=quarterly_balance_columns).rename(
    columns={"period": "balancePeriod"}
)
quarterly_cash_flow_dcf_inputs = quarterly_cash_flow_history.reindex(columns=quarterly_cash_flow_columns).rename(
    columns={
        "period": "cashFlowPeriod",
        "depreciationAndAmortization": "cashFlowDepreciationAndAmortization",
    }
)

quarterly_historical_dcf_inputs = (
    quarterly_income_dcf_inputs
    .merge(quarterly_balance_dcf_inputs, on=["date", "calendarYear"], how="outer")
    .merge(quarterly_cash_flow_dcf_inputs, on=["date", "calendarYear"], how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

quarterly_numeric_columns = [
    column
    for column in quarterly_historical_dcf_inputs.columns
    if column not in {"date", "incomePeriod", "balancePeriod", "cashFlowPeriod"}
    and quarterly_historical_dcf_inputs[column].dtype == object
]
for column in quarterly_numeric_columns:
    quarterly_historical_dcf_inputs[column] = pd.to_numeric(
        quarterly_historical_dcf_inputs[column], errors="coerce"
    )

quarterly_historical_dcf_inputs["operatingWorkingCapital"] = (
    quarterly_historical_dcf_inputs["netReceivables"].fillna(0)
    + quarterly_historical_dcf_inputs["inventory"].fillna(0)
    + quarterly_historical_dcf_inputs["otherCurrentAssets"].fillna(0)
    - quarterly_historical_dcf_inputs["accountPayables"].fillna(0)
    - quarterly_historical_dcf_inputs["otherCurrentLiabilities"].fillna(0)
)
quarterly_historical_dcf_inputs["changeInOperatingWorkingCapitalYoY"] = (
    quarterly_historical_dcf_inputs["operatingWorkingCapital"].diff(4)
)
quarterly_historical_dcf_inputs["cashFlowDepreciationAndAmortization"] = quarterly_historical_dcf_inputs[
    "cashFlowDepreciationAndAmortization"
].fillna(0)
quarterly_historical_dcf_inputs["capitalExpenditure"] = quarterly_historical_dcf_inputs[
    "capitalExpenditure"
].fillna(0)
quarterly_historical_dcf_inputs["operatingCashFlow"] = quarterly_historical_dcf_inputs[
    "operatingCashFlow"
].fillna(0)
quarterly_historical_dcf_inputs["freeCashFlow"] = quarterly_historical_dcf_inputs["freeCashFlow"].fillna(
    quarterly_historical_dcf_inputs["operatingCashFlow"] + quarterly_historical_dcf_inputs["capitalExpenditure"]
)

quarterly_ttm_flow_columns = [
    "revenue",
    "ebitda",
    "operatingIncome",
    "incomeBeforeTax",
    "incomeTaxExpense",
    "netIncome",
    "cashFlowDepreciationAndAmortization",
    "stockBasedCompensation",
    "operatingCashFlow",
    "capitalExpenditure",
    "freeCashFlow",
]
quarterly_ttm_dcf_inputs = quarterly_historical_dcf_inputs.copy()
for column in quarterly_ttm_flow_columns:
    quarterly_ttm_dcf_inputs[f"ttm{column[0].upper()}{column[1:]}"] = (
        quarterly_ttm_dcf_inputs[column].rolling(4, min_periods=4).sum()
    )

quarterly_ttm_dcf_inputs["ttmEffectiveTaxRate"] = (
    quarterly_ttm_dcf_inputs["ttmIncomeTaxExpense"]
    .div(quarterly_ttm_dcf_inputs["ttmIncomeBeforeTax"].replace(0, pd.NA))
    .clip(lower=0, upper=1)
    .fillna(0)
)
quarterly_ttm_dcf_inputs["ttmEbitAfterTax"] = quarterly_ttm_dcf_inputs["ttmOperatingIncome"] * (
    1 - quarterly_ttm_dcf_inputs["ttmEffectiveTaxRate"]
)
quarterly_ttm_dcf_inputs["ttmUnleveredFreeCashFlowProxy"] = (
    quarterly_ttm_dcf_inputs["ttmEbitAfterTax"]
    + quarterly_ttm_dcf_inputs["ttmCashFlowDepreciationAndAmortization"]
    - quarterly_ttm_dcf_inputs["ttmCapitalExpenditure"].abs()
    - quarterly_ttm_dcf_inputs["changeInOperatingWorkingCapitalYoY"].fillna(0)
)

In [ ]:
# 12. Build quarterly backfilled DCF and plot
quarterly_forecast_years = forecast_years if "forecast_years" in globals() else 5

quarterly_dcf_base = pd.merge_asof(
    quarterly_ttm_dcf_inputs.loc[quarterly_ttm_dcf_inputs["ttmRevenue"].notna()].sort_values("date"),
    dcf_full_price_history[["date", "price"]].sort_values("date"),
    on="date",
    direction="backward",
).rename(columns={"price": "marketPriceAtDate"})

quarterly_historical_dcf_rows = []

for index, row in quarterly_dcf_base.iterrows():
    if pd.isna(row["ttmRevenue"]) or row["ttmRevenue"] <= 0:
        continue

    lookback = quarterly_dcf_base.iloc[: index + 1].tail(8).copy()
    revenue_growth_series = pd.to_numeric(
        lookback["ttmRevenue"].pct_change(4).replace([float("inf"), float("-inf")], pd.NA),
        errors="coerce",
    )
    operating_margin_series = lookback["ttmOperatingIncome"].div(
        lookback["ttmRevenue"].replace(0, pd.NA)
    )
    depreciation_margin_series = lookback["ttmCashFlowDepreciationAndAmortization"].div(
        lookback["ttmRevenue"].replace(0, pd.NA)
    )
    capex_margin_series = lookback["ttmCapitalExpenditure"].abs().div(
        lookback["ttmRevenue"].replace(0, pd.NA)
    )
    working_capital_margin_series = lookback["operatingWorkingCapital"].div(
        lookback["ttmRevenue"].replace(0, pd.NA)
    )

    revenue_growth = clamp(median_or_default(revenue_growth_series, 0.05, window=4), -0.05, 0.2)
    operating_margin = clamp(
        median_or_default(
            operating_margin_series,
            row["ttmOperatingIncome"] / row["ttmRevenue"],
            window=4,
        ),
        -0.05,
        0.5,
    )
    depreciation_margin = clamp(
        median_or_default(depreciation_margin_series, 0.04, window=4), 0.0, 0.2
    )
    capex_margin = clamp(median_or_default(capex_margin_series, 0.04, window=4), 0.0, 0.2)
    working_capital_margin = median_or_default(
        working_capital_margin_series,
        row["operatingWorkingCapital"] / row["ttmRevenue"],
        window=4,
    )
    effective_tax_rate = clamp(
        median_or_default(lookback["ttmEffectiveTaxRate"], reference_risk_free_rate, window=4),
        0.0,
        0.35,
    )

    diluted_shares = row["weightedAverageShsOutDil"]
    if pd.isna(diluted_shares) or diluted_shares <= 0:
        diluted_shares = row["weightedAverageShsOut"]
    if pd.isna(diluted_shares) or diluted_shares <= 0:
        continue

    market_price = row["marketPriceAtDate"]
    market_equity_value = market_price * diluted_shares if pd.notna(market_price) else pd.NA
    total_debt = row["totalDebt"] if pd.notna(row["totalDebt"]) else 0.0
    net_debt = row["netDebt"] if pd.notna(row["netDebt"]) else total_debt
    debt_weight = (
        float(total_debt) / float(total_debt + market_equity_value)
        if pd.notna(market_equity_value) and total_debt + market_equity_value > 0
        else 0.0
    )
    equity_weight = 1 - debt_weight

    after_tax_cost_of_debt = reference_cost_of_debt * (1 - effective_tax_rate)
    wacc = debt_weight * after_tax_cost_of_debt + equity_weight * reference_cost_of_equity
    if not wacc or pd.isna(wacc):
        wacc = reference_cost_of_equity
    terminal_growth_rate = min(reference_terminal_growth_rate, wacc - 0.01) if wacc > 0.02 else 0.01
    terminal_growth_rate = max(terminal_growth_rate, 0.005)

    projected_revenue = float(row["ttmRevenue"])
    projected_working_capital = float(
        row["operatingWorkingCapital"] if pd.notna(row["operatingWorkingCapital"]) else 0.0
    )
    discounted_cash_flows = []
    projected_ufcf_final_year = pd.NA

    for forecast_year in range(1, quarterly_forecast_years + 1):
        projected_revenue *= 1 + revenue_growth
        projected_ebit = projected_revenue * operating_margin
        projected_ebit_after_tax = projected_ebit * (1 - effective_tax_rate)
        projected_depreciation = projected_revenue * depreciation_margin
        projected_capex = projected_revenue * capex_margin
        next_working_capital = projected_revenue * working_capital_margin
        projected_change_in_working_capital = next_working_capital - projected_working_capital
        projected_working_capital = next_working_capital
        projected_ufcf = (
            projected_ebit_after_tax
            + projected_depreciation
            - projected_capex
            - projected_change_in_working_capital
        )
        discounted_cash_flows.append(projected_ufcf / ((1 + wacc) ** forecast_year))
        projected_ufcf_final_year = projected_ufcf

    terminal_cash_flow = projected_ufcf_final_year * (1 + terminal_growth_rate)
    terminal_value = terminal_cash_flow / (wacc - terminal_growth_rate)
    present_terminal_value = terminal_value / ((1 + wacc) ** quarterly_forecast_years)
    enterprise_value = sum(discounted_cash_flows) + present_terminal_value
    equity_value = enterprise_value - net_debt
    quarterly_dcf_per_share = equity_value / diluted_shares
    premium_discount_pct = (
        quarterly_dcf_per_share / market_price - 1
        if pd.notna(market_price) and market_price
        else pd.NA
    )

    quarterly_historical_dcf_rows.append(
        {
            "date": row["date"],
            "calendarYear": row["calendarYear"],
            "incomePeriod": row["incomePeriod"],
            "ttmRevenue": row["ttmRevenue"],
            "ttmOperatingIncome": row["ttmOperatingIncome"],
            "ttmFreeCashFlow": row["ttmFreeCashFlow"],
            "ttmUnleveredFreeCashFlowProxy": row["ttmUnleveredFreeCashFlowProxy"],
            "revenueGrowthAssumption": revenue_growth,
            "operatingMarginAssumption": operating_margin,
            "taxRateAssumption": effective_tax_rate,
            "wacc": wacc,
            "terminalGrowthRate": terminal_growth_rate,
            "marketPriceAtDate": market_price,
            "enterpriseValue": enterprise_value,
            "equityValue": equity_value,
            "quarterlyBackfilledDcfPerShare": quarterly_dcf_per_share,
            "premiumDiscountPct": premium_discount_pct,
        }
    )

quarterly_historical_dcf_series = pd.DataFrame(quarterly_historical_dcf_rows)

quarterly_historical_dcf_plot = go.Figure()
quarterly_historical_dcf_plot.add_trace(
    go.Scatter(
        x=dcf_full_price_history["date"],
        y=dcf_full_price_history["price"],
        name=f"{SYMBOL} close",
        line={"color": "#e5eefc", "width": 2.5},
    )
)
quarterly_historical_dcf_plot.add_trace(
    go.Scatter(
        x=quarterly_historical_dcf_series["date"],
        y=quarterly_historical_dcf_series["quarterlyBackfilledDcfPerShare"],
        name="Quarterly backfilled DCF per share",
        mode="lines+markers",
        line={"color": "#22d3ee", "width": 2.5},
        marker={"color": "#22d3ee", "size": 7},
    )
)

quarterly_historical_dcf_plot.update_layout(
    title=f"{chart_label} quarterly backfilled DCF vs price",
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    xaxis_title="Date",
    yaxis_title="USD per share",
    autosize=True,
    height=680,
    margin={"l": 60, "r": 30, "t": 90, "b": 60},
)
quarterly_historical_dcf_plot.update_xaxes(
    showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True
)
quarterly_historical_dcf_plot.update_yaxes(
    showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True
)

quarterly_historical_dcf_plot.show(config={"responsive": True, "displaylogo": False})

## 13. Annual vs Quarterly DCF Comparison

> This section overlays the annual backfilled DCF and quarterly TTM backfilled DCF on the same chart so you can compare how much more responsive the quarterly build is relative to the annual build.

In [ ]:
# 14. Plot annual vs quarterly DCF comparison
annual_vs_quarterly_dcf = go.Figure()

annual_vs_quarterly_dcf.add_trace(
    go.Scatter(
        x=dcf_full_price_history["date"],
        y=dcf_full_price_history["price"],
        name=f"{SYMBOL} close",
        line={"color": "#e5eefc", "width": 2.5},
    )
)

annual_vs_quarterly_dcf.add_trace(
    go.Scatter(
        x=historical_dcf_series["date"],
        y=historical_dcf_series["historicalDcfPerShare"],
        name="Annual backfilled DCF",
        mode="lines+markers",
        line={"color": "#f472b6", "width": 2.5, "dash": "dash"},
        marker={"color": "#f472b6", "size": 8},
    )
)

annual_vs_quarterly_dcf.add_trace(
    go.Scatter(
        x=quarterly_historical_dcf_series["date"],
        y=quarterly_historical_dcf_series["quarterlyBackfilledDcfPerShare"],
        name="Quarterly TTM backfilled DCF",
        mode="lines+markers",
        line={"color": "#22d3ee", "width": 2.5},
        marker={"color": "#22d3ee", "size": 6},
    )
)

annual_vs_quarterly_dcf.update_layout(
    title=f"{chart_label} annual vs quarterly backfilled DCF",
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    xaxis_title="Date",
    yaxis_title="USD per share",
    autosize=True,
    height=680,
    margin={"l": 60, "r": 30, "t": 90, "b": 60},
)
annual_vs_quarterly_dcf.update_xaxes(
    showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True
)
annual_vs_quarterly_dcf.update_yaxes(
    showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True
)

annual_vs_quarterly_dcf.show(config={"responsive": True, "displaylogo": False})